# Forschungsfrage 4 - Semantische Heterogenität: LLM-basierter Ansatz (Claude)

Claude ordnet `thread_category` + `title` semantisch einer der 12 kanonischen
`parent_category`-Klassen zu - ohne trainierten Klassifikator, rein über
Sprachverständnis (vgl. Abschnitt 3.3.4). Bewertung auf demselben `test`-Split wie
in den beiden klassischen Notebooks. Zusätzlich wird die reale Anwendungsmenge
(500 Zeilen mit fehlender `parent_category`) mitverarbeitet, als Vorarbeit für
`Finale_Bereinigung_Rohdatensatz.ipynb`.

**Voraussetzung:** `ANTHROPIC_API_KEY` als Umgebungsvariable gesetzt.

**API-Hinweis:** `temperature` entfernt, stattdessen `thinking={"type": "disabled"}`
(vgl. Abschnitt 3.4.4). **Checkpoint/Resume:** 124 (Test) + 500 (Anwendung) = 624
Zeilen gesamt, jede einzeln sofort checkpointet.


**Hinweis zur Ausführung dieses Notebooks:** Der eigentliche API-Experiment-Lauf
wurde aus Stabilitätsgründen (lange Laufzeit, Checkpoint/Resume-Mechanismus über
mehrere Sitzungen hinweg) nicht interaktiv in Jupyter, sondern über ein separates
Runner-Skript (`tf*_llm_runner_script.py`) direkt im Terminal auf dem Rechner mit
gesetztem `ANTHROPIC_API_KEY` ausgeführt. Der vollständige Checkpoint dieses Laufs
liegt in `results/` und wird beim Ausführen dieses Notebooks automatisch geladen:
Da alle Zeilen bereits verarbeitet sind, überspringt die Checkpoint-Logik jede
Zeile, sodass **keine neuen API-Aufrufe** getätigt werden - die unten gezeigten
Zellen-Ausgaben sind somit die identischen, bereits im Terminal erzeugten
Originalergebnisse, hier lediglich zur Dokumentation und Nachvollziehbarkeit
erneut im Notebook ausgeführt.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os

from anthropic import Anthropic, RateLimitError, APIError
from sklearn.metrics import accuracy_score, f1_score

os.makedirs("results", exist_ok=True)

MODELL_NAME = "claude-sonnet-5"
# Preise geprueft am 31.08.2026 gegen https://platform.claude.com/docs/en/about-claude/pricing
PREIS_PRO_MTOK_INPUT = 2.00
PREIS_PRO_MTOK_OUTPUT = 10.00

TIME_BUDGET_SEC = float(os.environ.get("LLM_TIME_BUDGET_SEC", "1e9"))

client = Anthropic()
eval_df = pd.read_csv("benchmark/tf4_semantic_eval.csv")
app_df = pd.read_csv("benchmark/tf4_application_set.csv")
test = eval_df[eval_df["split"] == "test"].reset_index(drop=True)

VALID_CATEGORIES = sorted(eval_df["true_parent_category"].unique().tolist())
print(f"Anthropic-Client initialisiert. Modell: {MODELL_NAME}. Test: {len(test)}, Anwendung: {len(app_df)}")


Anthropic-Client initialisiert. Modell: claude-sonnet-5. Test: 124, Anwendung: 500


## 2. Prompt-Design, API-Hilfsfunktion und Checkpoint-Verarbeitungsschleife

In [2]:
SYSTEM_PROMPT = f"""Du bist ein Experte für Produktkategorisierung in einem Online-Deal-Forum. \
Ordne die gezeigte Zeile (Unterkategorie thread_category und Titel) GENAU EINER der folgenden \
zulässigen Hauptkategorien zu:

{VALID_CATEGORIES}

Antworte AUSSCHLIESSLICH mit einem JSON-Objekt:
{{"parent_category": "<eine der zulässigen Kategorien>", "reasoning": "kurze Begründung"}}
"""

def build_prompt(row):
    return f'thread_category: "{row["thread_category"]}"\ntitle: "{row["title"]}"'

def rufe_claude_api_mit_backoff(system_prompt, user_prompt, model, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model, max_tokens=150, system=system_prompt,
                messages=[{"role": "user", "content": user_prompt}],
                thinking={"type": "disabled"},
            )
            raw_text = response.content[0].text.strip()
            usage = {"input_tokens": response.usage.input_tokens, "output_tokens": response.usage.output_tokens}
            start, end = raw_text.find("{"), raw_text.rfind("}")
            if start == -1 or end == -1 or end <= start:
                raise json.JSONDecodeError("JSON nicht extrahierbar.", raw_text, 0)
            return json.loads(raw_text[start:end + 1]), usage
        except RateLimitError:
            delay = 2 ** attempt
            print(f"  Ratenlimit, warte {delay}s (Versuch {attempt + 1}/{max_retries})...")
            time.sleep(delay)
        except APIError as e:
            print(f"  API-Fehler (Versuch {attempt + 1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(5)
            else:
                return None, None
        except (json.JSONDecodeError, IndexError, KeyError) as e:
            print(f"  Antwort nicht verwertbar: {e}")
            return None, None
    return None, None


def process_loop_mit_checkpoint(items_df, id_col, checkpoint_path, timing_state, timing_key, label):
    if os.path.exists(checkpoint_path):
        done = pd.read_csv(checkpoint_path)
        done_ids = set(done[id_col].tolist())
        print(f"  Checkpoint gefunden: {len(done_ids)} von {len(items_df)} {label}-Zeilen bereits verarbeitet.")
    else:
        done = pd.DataFrame(columns=[id_col])
        done_ids = set()

    t0 = time.time()
    budget_exceeded = False
    for i, (_, row) in enumerate(items_df.iterrows()):
        if row[id_col] in done_ids:
            continue
        if time.time() - t0 > TIME_BUDGET_SEC:
            budget_exceeded = True
            print(f"  Zeitbudget ({TIME_BUDGET_SEC:.0f}s) erreicht, breche {label} kontrolliert ab.")
            break
        result, usage = rufe_claude_api_mit_backoff(SYSTEM_PROMPT, build_prompt(row), MODELL_NAME)
        if result is not None:
            pred = result.get("parent_category")
            if pred not in VALID_CATEGORIES:
                pred = None
            neue_zeile = {id_col: row[id_col], "parent_category_pred_llm": pred, "reasoning": result.get("reasoning", ""),
                          "input_tokens": usage["input_tokens"], "output_tokens": usage["output_tokens"]}
        else:
            neue_zeile = {id_col: row[id_col], "parent_category_pred_llm": None, "reasoning": "[FEHLER/FALLBACK - kein gueltiges LLM-Ergebnis]",
                          "input_tokens": 0, "output_tokens": 0}
        done = pd.concat([done, pd.DataFrame([neue_zeile])], ignore_index=True)
        done.to_csv(checkpoint_path, index=False)
        if len(done) % 50 == 0:
            print(f"  {label}: {len(done)}/{len(items_df)}")

    timing_state[timing_key] = timing_state.get(timing_key, 0.0) + (time.time() - t0)

    if budget_exceeded or len(done) < len(items_df):
        raise RuntimeError(f"{label}-Aufgabe unvollstaendig ({len(done)}/{len(items_df)}). Notebook erneut ausfuehren.")

    total_in = int(done["input_tokens"].sum())
    total_out = int(done["output_tokens"].sum())
    return done, total_in, total_out, timing_state[timing_key]


TIMING_STATE_PATH = "results/tf4_llm_timing_state.json"
if os.path.exists(TIMING_STATE_PATH):
    with open(TIMING_STATE_PATH) as f:
        timing_state = json.load(f)
else:
    timing_state = {}


## 3. Verarbeitung: Test-Split (Bewertung)

In [3]:
results_df, total_in, total_out, wall_time = process_loop_mit_checkpoint(
    test, "row_id", "results/tf4_llm_test_checkpoint.csv", timing_state, "test_sec", "test")

with open(TIMING_STATE_PATH, "w") as f:
    json.dump(timing_state, f, indent=2)

merged = results_df.merge(test[["row_id", "true_parent_category"]], on="row_id")
valid = merged.dropna(subset=["parent_category_pred_llm"])

accuracy = accuracy_score(valid["true_parent_category"], valid["parent_category_pred_llm"])
macro_f1 = f1_score(valid["true_parent_category"], valid["parent_category_pred_llm"], average="macro")
print(f"Accuracy: {accuracy:.3f}  Macro-F1: {macro_f1:.3f}  (n_valid={len(valid)}/{len(merged)}, kumulierte Laufzeit {wall_time:.1f}s)")

merged.to_csv("results/tf4_llm_predictions.csv", index=False)


  Checkpoint gefunden: 124 von 124 test-Zeilen bereits verarbeitet.
Accuracy: 0.887  Macro-F1: 0.808  (n_valid=124/124, kumulierte Laufzeit 304.3s)


## 4. Verarbeitung: Anwendungsmenge (reale Lücken, für Kapitel 4.6)

In [4]:
app_results_df, total_in_app, total_out_app, wall_time_app = process_loop_mit_checkpoint(
    app_df, "row_id", "results/tf4_llm_application_checkpoint.csv", timing_state, "application_sec", "Anwendung")

with open(TIMING_STATE_PATH, "w") as f:
    json.dump(timing_state, f, indent=2)

app_out = app_df[["row_id", "thread_category", "title"]].merge(
    app_results_df[["row_id", "parent_category_pred_llm"]], on="row_id")
app_out.to_csv("results/tf4_llm_application_predictions.csv", index=False)
print(f"Anwendungsmenge verarbeitet: {len(app_out)} Zeilen (kumulierte Laufzeit {wall_time_app:.1f}s)")


  Checkpoint gefunden: 500 von 500 Anwendung-Zeilen bereits verarbeitet.
Anwendungsmenge verarbeitet: 500 Zeilen (kumulierte Laufzeit 1259.4s)


## 5. Metriken und Laufzeit-/Kosten-Log speichern

In [5]:
cost_test = total_in/1_000_000*PREIS_PRO_MTOK_INPUT + total_out/1_000_000*PREIS_PRO_MTOK_OUTPUT
cost_app = total_in_app/1_000_000*PREIS_PRO_MTOK_INPUT + total_out_app/1_000_000*PREIS_PRO_MTOK_OUTPUT

metrics = {
    "experiment": "TF4_Semantik", "method": "Claude_LLM", "model": MODELL_NAME,
    "accuracy": accuracy, "macro_f1": macro_f1, "n_valid": len(valid),
    "wall_time_sec": wall_time, "estimated_cost_usd": cost_test + cost_app,
    "n_application_rows": len(app_df),
}
with open("results/tf4_llm_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

log_rows = pd.DataFrame([{
    "experiment": "TF4_Semantik", "method": "Claude_LLM", "n_items": len(test),
    "wall_time_sec": wall_time, "input_tokens": total_in, "output_tokens": total_out,
    "estimated_cost_usd": cost_test, "model_name": MODELL_NAME,
}])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_rows["experiment"], log_rows["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    _combined_log = pd.concat([_old_log, log_rows], ignore_index=True)
else:
    _combined_log = log_rows
_combined_log.to_csv(log_path, index=False)

print(f"Geschaetzte Gesamtkosten TF4 (LLM, Test+Anwendung): ${cost_test + cost_app:.4f}")
print("Gespeichert: results/tf4_llm_predictions.csv, results/tf4_llm_application_predictions.csv, results/tf4_llm_metrics.json")
print("NOTEBOOK_EXECUTED_OK")


Geschaetzte Gesamtkosten TF4 (LLM, Test+Anwendung): $0.9394
Gespeichert: results/tf4_llm_predictions.csv, results/tf4_llm_application_predictions.csv, results/tf4_llm_metrics.json
NOTEBOOK_EXECUTED_OK
